<a href="https://colab.research.google.com/github/priyal6/AI-AGENT/blob/main/reactagent_with_mem0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
pip install langchain langchain-openai mem0ai

In [27]:
pip install langgraph langchain-core langchain-openai

In [31]:
import os
from langchain_openai import ChatOpenAI
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["MEM0_API_KEY"] = userdata.get('MEM0_API_KEY')

llm = ChatOpenAI(model="gpt-4o-mini")

In [32]:
from langchain.tools import tool

@tool
def call_fn(name: str) -> str:
    """Call the provided person by name."""
    return f"Calling... {name}"

@tool
def email_fn(name: str) -> str:
    """Email the provided person by name."""
    return f"Emailing... {name}"

@tool
def order_food(name: str, dish: str) -> str:
    """Order food for a person. Args: name and dish."""
    return f"Ordering {dish} for {name}"

tools = [call_fn, email_fn, order_food]

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [33]:
from mem0 import MemoryClient

mem0_client = MemoryClient(api_key=os.environ["MEM0_API_KEY"])
user_id = "david"

In [34]:
from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

def chat_with_memory(user_message: str, user_id: str):
    memories = mem0_client.search(user_message, filters={"user_id": user_id})
    memory_text = "\n".join(m["memory"] for m in memories.get("results", [])) or "No past memories."

    system_prompt = f"""You are a helpful assistant.

Relevant memories from past interactions:
{memory_text}

Use these memories to personalize your responses."""

    agent = create_react_agent(llm, tools, prompt=system_prompt)
    response = agent.invoke({"messages": [{"role": "user", "content": user_message}]})
    output = response["messages"][-1].content

    mem0_client.add([
        {"role": "user", "content": user_message},
        {"role": "assistant", "content": output}
    ], user_id=user_id)

    return output

In [35]:
print(chat_with_memory("Hi, my name is David", user_id))
print(chat_with_memory("I love pizza on weekends", user_id))
print(chat_with_memory("My preferred contact is email", user_id))

# Later — agent recalls preferences automatically:
print(chat_with_memory("I'm hungry, order me something and send the bill", user_id))

/tmp/ipykernel_454/1612269642.py:17: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools, prompt=system_prompt)


Hello, David! How can I assist you today?
That's great to hear, David! Would you like me to help you order some pizza for this weekend?
Got it, David! I'll make sure to contact you via email from now on. If there's anything else you'd like to add, just let me know!
I've ordered a delicious pizza for you, David, and I've sent the bill to your email. Enjoy your meal!
